<div align="center">

<a href="https://colab.research.google.com/github/utkukose/human-computer-interaction-NB-lecture/blob/main/exams/final/FIN1_assistant_that_suggests.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

# Final option 1: An assistant that suggests

**Human-Computer Interaction (BIL-429 İnsan-Bilgisayar Etkileşimi)**<br>Final project: An interface with an artificial intelligence (AI) feature, designed and evaluated. Starter Colab notebook

[Project page](https://github.com/utkukose/human-computer-interaction-NB-lecture/tree/main/exams/final) | [Course repository](https://github.com/utkukose/human-computer-interaction-NB-lecture)

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

<sub>Content licensed under CC BY 4.0, code under MIT. This course is updated in line with current developments in the field. First version: November 2026.</sub>

</div>

## How to use this starter Colab notebook

Save a copy in Drive first, with File, Save a copy in Drive. Every code cell runs as it is with an example case: An expense app whose assistant suggests the category of a receipt. Replace the values marked `# TODO` with the measurements of your own interface. Then run all cells again from the top, with Runtime, Run all, and use the printed values and figures in the report. Each section names the week and the lecture section that explain its method. The last section collects the numbers for the results table of the report. Keep your own data in the cells, so that the Colab notebook documents how every number of the report was obtained.

## 0. Setup

The cell loads the libraries and creates the dictionary `RESULTS`, which collects the numbers of the report.

In [ ]:
import math
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.linear_model import LogisticRegression

RANDOM_STATE = 42
RESULTS = {}            # the numbers for the results table of the report
print("Setup complete.")

## 1. The feature and its log

*Lecture: [Week 13, Trust and reliance](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-13/lecture.html#trust-and-reliance)*

The evaluation of a feature of artificial intelligence (AI) starts with a log of its suggestions. For each case, the log holds the confidence that the feature shows and whether the suggestion was right. In the example, an expense app suggests the category of a receipt from its amount, its hour and the day of the week. Replace the simulated log with the log of your own feature, with at least 200 cases whose right answer you know.

In [ ]:
CATS = ["food", "transport", "office", "travel", "health"]
MU = [2.6, 2.2, 3.4, 4.6, 3.0]           # mean log amount of each category
HOUR = [13, 9, 11, 15, 17]               # mean hour of each category
WEEKEND = [0.4, 0.2, 0.05, 0.3, 0.2]     # share of receipts at weekends
rng = np.random.default_rng(RANDOM_STATE)

def receipts(n):
    """Simulated receipts: log amount, hour, weekend flag and the category."""
    y = rng.integers(0, len(CATS), n)
    X = np.column_stack([rng.normal(np.take(MU, y), 0.6),
                         rng.normal(np.take(HOUR, y), 3.0),
                         rng.random(n) < np.take(WEEKEND, y)]).astype(float)
    return X, y

X_tr, y_tr = receipts(2000)
X_te, y_te = receipts(1000)
model = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
P = model.predict_proba(X_te)
# TODO: replace LOG by the log of your feature: confidence and right (1 or 0)
LOG = pd.DataFrame({"confidence": P.max(axis=1),
                    "right": (P.argmax(axis=1) == y_te).astype(int)})
print(f"cases {len(LOG)}, accuracy {LOG.right.mean():.3f}, "
      f"mean confidence {LOG.confidence.mean():.3f}")
RESULTS["accuracy of the AI feature"] = round(LOG.right.mean(), 3)

## 2. Is the confidence honest?

*Lecture: [Week 13, Confidence and calibration](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-13/lecture.html#confidence-and-calibration)*

A calibrated feature is right for a share $c$ of the suggestions that it shows with the confidence $c$. A reliability table sorts the suggestions into ten bins of confidence and compares the share right $a_b$ of each bin with its mean confidence $\bar c_b$. The expected calibration error (ECE) is $\mathrm{ECE} = \sum_b \frac{n_b}{N}\,|a_b - \bar c_b|$. The Brier score of the confidence, $\frac{1}{N}\sum_i (c_i - o_i)^2$ with $o_i = 1$ for a right suggestion, rewards confidences that are both honest and sharp.

In [ ]:
def reliability(c, ok, bins=10):
    """Reliability table: count, mean confidence and share right per bin."""
    c, ok = np.asarray(c, float), np.asarray(ok, float)
    b = np.minimum((c * bins).astype(int), bins - 1)
    rows = [(k, int((b == k).sum()), c[b == k].mean(), ok[b == k].mean())
            for k in range(bins) if (b == k).any()]
    return pd.DataFrame(rows, columns=["bin", "n", "confidence", "right"])

def ece(c, ok, bins=10):
    """Expected calibration error: the weighted gap of the bins."""
    t = reliability(c, ok, bins)
    return float((t.n / t.n.sum() * (t.right - t.confidence).abs()).sum())

table = reliability(LOG.confidence, LOG.right)
print(table.round(3).to_string(index=False))
brier_c = float(((LOG.confidence - LOG.right) ** 2).mean())
print(f"ECE {ece(LOG.confidence, LOG.right):.4f}, "
      f"Brier score of the confidence {brier_c:.4f}")
RESULTS["expected calibration error"] = round(ece(LOG.confidence, LOG.right), 4)
RESULTS["Brier score of the confidence"] = round(brier_c, 4)

In [ ]:
# Figure: the reliability diagram of the feature
plt.figure(figsize=(5.2, 4.2))
plt.bar(table.confidence, table.right, width=0.07, color="#5A4BB5", alpha=0.8)
plt.plot([0, 1], [0, 1], "--", color="#585E6E")
plt.xlabel("confidence of the suggestion")
plt.ylabel("share right")
plt.title("bars below the diagonal: overconfident")
plt.tight_layout()
plt.show()

## 3. Reliance with a threshold

*Lecture: [Week 13, Trust and reliance](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-13/lecture.html#trust-and-reliance)*

The interface can show a suggestion only when its confidence reaches a threshold $\tau$, and let the user decide alone otherwise. With the share $s$ of the suggestions at or above $\tau$, their accuracy $a_{\text{hi}}$ and the accuracy $h$ of a user who decides alone, the team is right with $A = s\,a_{\text{hi}} + (1 - s)\,h$. Measure $h$ with a few users who categorise receipts without the assistant.

In [ ]:
H_ALONE = 0.80        # TODO: accuracy of a user who decides alone

def team_accuracy(tau, h=H_ALONE):
    """Show the suggestions with confidence >= tau; the user decides the rest."""
    hi = LOG.confidence >= tau
    s = hi.mean()
    a_hi = LOG.right[hi].mean() if hi.any() else 0.0
    return s * a_hi + (1 - s) * h, s, a_hi

taus = np.round(np.arange(0.0, 1.0, 0.05), 2)
team = pd.DataFrame([(t, *team_accuracy(t)) for t in taus],
                    columns=["tau", "team", "shown", "right among shown"])
best = team.loc[team.team.idxmax()]
print(team.round(3).to_string(index=False))
print(f"best threshold {best.tau}: team {best.team:.3f}, AI alone "
      f"{LOG.right.mean():.3f}, user alone {H_ALONE}")
RESULTS["threshold of the redesign"] = float(best.tau)
RESULTS["team accuracy at the threshold"] = round(float(best.team), 3)

## 4. A usability test of two designs

*Lecture: [Week 9, Completion rates](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-09/lecture.html#completion-rates) and [Week 9, Time on task](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-09/lecture.html#time-on-task) and [Week 9, Comparing designs](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-09/lecture.html#comparing-designs)*

Design A shows the suggestion alone, and design B adds its confidence and the threshold of Section 3. Each participant files a set of receipts, some with a wrong suggestion, and fills in the System Usability Scale (SUS) at the end. The completion rate gets an adjusted Wald interval, and the task times a geometric mean with an interval from the logarithms. Welch's test compares the designs. Overreliance is the share of the wrong suggestions that a participant accepted. The example simulates 12 participants per design; a test with real participants asks for their consent and keeps the data anonymous.

In [ ]:
def sus_score(answers):
    """SUS score of one questionnaire: ten answers from 1 to 5, statement 1 first."""
    total = 0
    for j, x in enumerate(answers, start=1):
        total += x - 1 if j % 2 == 1 else 5 - x
    return 2.5 * total

def adjusted_wald(x, n, z=1.96):
    """Adjusted Wald interval of a completion rate: (p_adj, low, high)."""
    p_adj = (x + z ** 2 / 2) / (n + z ** 2)
    margin = z * math.sqrt(p_adj * (1 - p_adj) / (n + z ** 2))
    return p_adj, max(0.0, p_adj - margin), min(1.0, p_adj + margin)

def geo_mean_ci(times, conf=0.95):
    """Geometric mean of task times with its t interval from the logarithms."""
    logs = np.log(np.asarray(times, dtype=float))
    n, mean, sd = len(logs), logs.mean(), logs.std(ddof=1)
    t = stats.t.ppf(0.5 + conf / 2, n - 1)
    margin = t * sd / math.sqrt(n)
    return math.exp(mean), math.exp(mean - margin), math.exp(mean + margin)

In [ ]:
# TODO: replace the simulated sessions with your sessions, one row per person
rng = np.random.default_rng(RANDOM_STATE + 1)
rows = []
for design, t_med, p_done, p_over, sus_mean in [("A", 95, 0.75, 0.55, 3.4),
                                                ("B", 80, 0.92, 0.25, 3.9)]:
    for k in range(12):
        sus = np.clip(np.round(rng.normal(sus_mean, 0.6, 10)), 1, 5)
        sus[1::2] = 6 - sus[1::2]                 # even statements are negative
        rows.append(dict(design=design, participant=f"{design}{k + 1}",
                         completed=int(rng.random() < p_done),
                         time_s=round(t_med * math.exp(rng.normal(0, 0.3)), 1),
                         wrong_shown=4, wrong_accepted=int(rng.binomial(4, p_over)),
                         sus=sus_score(sus)))
SESSIONS = pd.DataFrame(rows)
for design, part in SESSIONS.groupby("design"):
    p, lo, hi = adjusted_wald(int(part.completed.sum()), len(part))
    gm, glo, ghi = geo_mean_ci(part.time_s[part.completed == 1])
    over = part.wrong_accepted.sum() / part.wrong_shown.sum()
    print(f"design {design}: completed {part.completed.sum()} of {len(part)} "
          f"({lo:.2f} to {hi:.2f}), time {gm:.1f} s ({glo:.1f} to {ghi:.1f}), "
          f"SUS {part.sus.mean():.1f}, overreliance {over:.3f}")
    RESULTS[f"design {design}: SUS"] = round(part.sus.mean(), 1)
    RESULTS[f"design {design}: overreliance"] = round(over, 3)
A, B = (SESSIONS[SESSIONS.design == d] for d in "AB")
p_time = stats.ttest_ind(np.log(A.time_s[A.completed == 1]),
                         np.log(B.time_s[B.completed == 1]), equal_var=False).pvalue
p_sus = stats.ttest_ind(A.sus, B.sus, equal_var=False).pvalue
fmt_p = lambda p: f"{p:.3f}" if p >= 0.001 else "< 0.001"
print(f"Welch's test: times on logarithms p = {fmt_p(p_time)}, "
      f"SUS p = {fmt_p(p_sus)}")
RESULTS["p of the times (Welch on logarithms)"] = fmt_p(p_time)
RESULTS["p of the SUS scores (Welch)"] = fmt_p(p_sus)

## 5. An accessibility audit of the AI feature

*Lecture: [Week 10, The Web Content Accessibility Guidelines](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-10/lecture.html#wcag) and [Week 10, Plain words](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-10/lecture.html#plain-words)*

The confidence and the explanation must reach every user. The Web Content Accessibility Guidelines (WCAG) set the rules. Their colours need a contrast ratio of at least 4.5:1 for text, and the buttons at least 24 by 24 pixels of Cascading Style Sheets (CSS). The explanation needs plain words. The Flesch-Kincaid grade level $0.39\,\frac{\text{words}}{\text{sentences}} + 11.8\,\frac{\text{syllables}}{\text{words}} - 15.59$ estimates the years of school needed to read a text.

In [ ]:
def contrast_ratio(c1, c2):
    """WCAG contrast ratio of two colours given as hex codes."""
    def lum(code):
        c = np.array([int(code.lstrip("#")[i:i + 2], 16) for i in (0, 2, 4)]) / 255
        lin = np.where(c <= 0.04045, c / 12.92, ((c + 0.055) / 1.055) ** 2.4)
        return float(lin @ np.array([0.2126, 0.7152, 0.0722]))
    L1, L2 = lum(c1), lum(c2)
    return (max(L1, L2) + 0.05) / (min(L1, L2) + 0.05)

def syllables_en(word):
    """Approximate English syllables: runs of vowels, minus a silent e."""
    w = re.sub(r"[^a-z]", "", word.lower())
    n = len(re.findall(r"[aeiouy]+", w))
    if n > 1 and w.endswith("e") and not w.endswith(("le", "ee")):
        n -= 1
    return max(1, n)

def fkgl(text):
    """Flesch-Kincaid grade level of an English text."""
    words = re.findall(r"[A-Za-z]+", text)
    sentences = [s for s in re.split(r"[.!?]+", text) if s.strip()]
    syll = sum(syllables_en(w) for w in words)
    return 0.39 * len(words) / len(sentences) + 11.8 * syll / len(words) - 15.59

BADGES = {"high": ("#2E8B57", "#FFFFFF"), "medium": ("#C98510", "#FFFFFF"),
          "low": ("#B53F2E", "#FFFFFF")}          # TODO: text and background
for name, (fg, bg) in BADGES.items():
    cr = contrast_ratio(fg, bg)
    print(f"{name} badge: {cr:.2f}:1 -> {'pass' if cr >= 4.5 else 'FAIL'} for text")
TARGETS_PX = {"accept": 44, "change category": 20}    # TODO: sizes in CSS px
for name, px in TARGETS_PX.items():
    print(f"{name}: {px} px -> {'pass' if px >= 24 else 'FAIL'}")
EXPLANATION = ("This receipt looks like travel because its amount is high. "
               "Similar receipts on weekday afternoons were mostly travel.")   # TODO
print(f"grade of the explanation: {fkgl(EXPLANATION):.1f}")
RESULTS["badges failing 4.5:1"] = sum(
    contrast_ratio(*v) < 4.5 for v in BADGES.values())
RESULTS["grade of the explanation"] = round(fkgl(EXPLANATION), 1)

## 6. Results for the report

The cell collects the numbers that the sections above stored in `RESULTS`. Copy them into the results table of the report, and explain each one in the text. A value that changes the design deserves a figure.

In [ ]:
import pandas as pd

print(pd.Series(RESULTS, name="value", dtype=object).to_string())

---

<div align="center">

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

This course is updated in line with current developments in the field. First version: November 2026.

</div>